# Benchmark tables and event-stream comparisons
Each dataset has a main table and a separate spatial voxel comparison. Chamfer Full is the only Chamfer variant shown. Edit TABLE_COLUMNS in the table cell and rerun that cell. The plots below compare one sequence using the benchmark crop and physical frame timestamps.

Each number is a sequence-weighted mean. Within a column, methods use the same sequences. The voxel table also matches sequences across available scales for each metric and polarity. Headers show the number of shared sequences. Only finite, completed scores are included. Fano scores with partial spatial support are included and flagged in the report inventory.

Blue marks the four best rank levels (darkest first, equal scores share a rank). For descriptive columns (including Same-ts), cells show the mean raw value and colors rank the mean absolute difference from ground truth across matched sequences. Ground truth is a reference row. Count ratio shows the mean ratio and, in parentheses, the mean per-sequence absolute deviation from 1 used for ranking. A dash indicates unavailable data. See the report inventory to check coverage and the selected runs.

Tables combine saved runs with matching recorded scoring parameters for each metric. Duplicate sequence/method rows prefer a finite completed score, then the newest CSV. RUN_OVERRIDE pins a single run.


In [ ]:
from pathlib import Path
import json
from html import escape

import numpy as np
import pandas as pd
from IPython.display import HTML, display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "benchmarks/run_metric.py").is_file()), Path.cwd())
REPORT_ROOT = root / "reports"
SPLIT = "test"
# Optional: choose a specific run directory name per (dataset, metric, block size).
# Use None for the block size of a non-voxel metric.
RUN_OVERRIDE = {}
DATASET = "bsergb"
SEQUENCE = None       # None selects the first sequence with a saved event stream.
METHODS = None        # None discovers all saved methods. Example: ["v2ce", "evtween_v5.2_student"]
START_FRAME = 0       # Zero-based index of the first frame in the detail view.
N_FRAMES = 17         # 17 frames span 16 frame intervals.
RATE_BIN_S = 0.001    # Event-rate bin width in seconds, including the detail view.
DATA_ROOT = None      # Optional path containing dataset/split/sequence folders.
PRED_ROOT = root / "data_test_evt"

RASTER_MAX_POINTS = 20_000  # Display cap per method in the selected frame window.
CLOUD_MAX_POINTS = 20_000   # Upper limit for the largest 3D cloud; None displays all events.
CLOUD_SEED = 0             # Deterministic display-only sampling, shared fraction across methods.


In [ ]:
MAIN = {
    "chamfer_full": "Chamfer Full ↓", "iei_distance": "IEI distance ↓",
    "fano_factor": "Fano error ↓", "event_count_ratio": "Count ratio ≈1",
    "event_count": "Event count", "event_rate": "Event rate (events/s)",
    "on_percentage": "ON (%)", "isi_median_ms": "Median ISI (ms)",
    "isi_cv": "ISI CV", "same_ts": "Same-ts fraction",
}
VOXEL = {"voxel_mae": "MAE", "voxel_rmse": "RMSE"}
CHANNELS = {"value": "Signed", "value_on": "ON", "value_off": "OFF"}
LOWER = {"chamfer_full", "iei_distance", "fano_factor", *VOXEL}
runs = {}
for cfg_path in sorted(REPORT_ROOT.glob("*/*/*/config.json")):
    cfg = json.loads(cfg_path.read_text())
    metric = cfg["metric"]
    if cfg["split"] != SPLIT or metric not in {*MAIN, *VOXEL}:
        continue
    size = cfg.get("options", {}).get("block_size", 1) if metric in VOXEL else None
    csv = cfg_path.with_name("results.csv")
    if csv.is_file():
        runs.setdefault((cfg["dataset"], metric, size), []).append(csv)

reports, inventory = {}, []
for key, paths in runs.items():
    override = RUN_OVERRIDE.get(key)
    choices = [p for p in paths if override is None or p.parent.name == override]
    if not choices:
        raise ValueError(f"Unknown run {override!r} for {key}")
    choices.sort(key=lambda p: (p.stat().st_mtime_ns, str(p)))
    def protocol(path):
        cfg = json.loads(path.with_name("config.json").read_text())
        opts = dict(cfg["options"])
        if key[1] == "chamfer_full":
            for field in ("device", "workers", "chunk_size", "backend_version"):
                opts.pop(field, None)
        return cfg["unit"], cfg["interval"], cfg["crop"], opts

    anchor = protocol(choices[-1])
    frames, sources = [], []
    for path in choices:
        if protocol(path) != anchor:
            print(f"Skipped different protocol: {path.parent}")
            continue
        frame = pd.read_csv(path)
        if frame.duplicated(["sequence", "method"]).any():
            raise ValueError(f"Duplicate sequence/method rows in {path}")
        sources.append(path.parent.name)
        frames.append(frame)
    raw = pd.concat(frames, ignore_index=True)
    # Prefer a completed score, then the latest CSV for the same sequence/method.
    usable = raw.status.eq("ok") | ((key[1] == "fano_factor") & raw.status.eq("partial_support"))
    raw["_usable"] = usable & np.isfinite(pd.to_numeric(raw.value, errors="coerce"))
    raw = (raw.sort_values("_usable", kind="stable")
              .drop_duplicates(["sequence", "method"], keep="last").drop(columns="_usable"))
    ok = raw.status.eq("ok") | ((key[1] == "fano_factor") & raw.status.eq("partial_support"))
    rows = raw.copy()
    for field in CHANNELS:
        rows[field] = pd.to_numeric(rows.get(field, np.nan), errors="coerce")
        rows.loc[~ok | ~np.isfinite(rows[field]), field] = np.nan
    reports[key] = rows
    inventory.append(dict(dataset=key[0], metric=key[1], block=key[2], run=', '.join(sources),
                          candidates=len(paths), scored=int(rows.value.notna().sum()), rows=len(raw),
                          partial_support=int(raw.status.eq("partial_support").sum())))
if not reports:
    print(f"No matching {SPLIT} reports under {REPORT_ROOT}")
if set(RUN_OVERRIDE) - set(reports):
    raise ValueError(f"Missing report overrides: {set(RUN_OVERRIDE) - set(reports)}")
datasets = sorted({key[0] for key in reports})
voxel_sizes = sorted({size for _, metric, size in reports if metric in VOXEL})
print(f"{SPLIT} | {', '.join(datasets)} | {len(reports)} selected reports")

if inventory:
    display(pd.DataFrame(inventory)[["dataset", "metric", "run", "scored", "rows"]])


In [ ]:
def summarize(dataset, metric, field, sizes):
    matrices, common = {}, None
    for size in sizes:
        rows = reports.get((dataset, metric, size))
        if rows is None:
            continue
        matrix = rows.pivot(index="sequence", columns="method", values=field)
        matrices[size] = matrix
        available = matrix.drop(columns="ground_truth", errors="ignore").dropna(axis=1, how="all")
        if available.shape[1]:
            complete = available.dropna().index
            common = complete if common is None else common.intersection(complete)
    columns = {}
    for size, matrix in matrices.items():
        paired = matrix.reindex([] if common is None else common)
        descriptor = metric in MAIN and metric not in LOWER and metric != "event_count_ratio"
        if descriptor:
            paired = paired.loc[paired["ground_truth"].notna()] if "ground_truth" in paired else paired.iloc[:0]
        means = paired.mean()
        ranks = pd.Series(dtype=float)
        deviation = None
        if metric == "event_count_ratio":
            deviation = paired.sub(1).abs().mean()
            ranks = deviation
        elif metric in LOWER:
            ranks = means
        elif descriptor and "ground_truth" in paired:
            ranks = paired.sub(paired["ground_truth"], axis="index").abs().mean()
        ranks = ranks.drop(labels="ground_truth", errors="ignore").rank(method="dense")
        columns[size] = (means, ranks, deviation, len(paired))
    return columns

# Precompute the summaries used by the tables.
tables = {}
for dataset in datasets:
    main, voxel = {}, {}
    for metric, label in MAIN.items():
        main[label] = summarize(dataset, metric, "value", [None]).get(None)
    for metric, short in VOXEL.items():
        for field, channel in CHANNELS.items():
            by_size = summarize(dataset, metric, field, voxel_sizes)
            for size in voxel_sizes:
                voxel[f"{short} {channel} {size}×{size}"] = by_size.get(size)
    tables[dataset] = main, voxel

In [ ]:
from fnmatch import fnmatchcase
# EDIT THIS LINE: use the printed labels or wildcards (e.g. "MAE ON *" or "RMSE Signed 4×4").
TABLE_COLUMNS = [*MAIN.values(), "MAE Signed *", "RMSE Signed *"]

def render(title, columns, selected, methods):
    labels = [label for label in columns if label in selected]
    if not labels:
        return
    palette = ["#164b80", "#75a9d6", "#b4d4ed", "#e0eef8"]
    html = [f"<h4>{escape(title)}</h4><div style='overflow-x:auto'><table><thead><tr><th>Method</th>"]
    for label in labels:
        col = columns[label]
        count = col[3] if col else 0
        html.append(f"<th>{escape(label)}<br><small>n={count}</small></th>")
    html.append("</tr></thead><tbody>")
    for method in methods:
        if method == "ground_truth" and not any(columns[l] and pd.notna(columns[l][0].get(method)) for l in labels):
            continue
        html.append(f"<tr><th>{escape(str(method))}</th>")
        for label in labels:
            col = columns[label]
            value = col[0].get(method, np.nan) if col else np.nan
            rank = col[1].get(method, np.nan) if col else np.nan
            style, number = "", "—"
            if pd.notna(value):
                number = f"{value:,.5g}"
                if col[2] is not None:
                    number += f" ({col[2][method]:.4g})"
                if pd.notna(rank) and rank <= 4:
                    color = "white" if rank == 1 else "#102b43"
                    style = f"background:{palette[int(rank)-1]};color:{color}"
            html.append(f"<td style='{style}'>{number}</td>")
        html.append("</tr>")
    display(HTML("".join(html) + "</tbody></table></div>"))

if tables:
    print("Available columns:", ", ".join(label for group in next(iter(tables.values())) for label in group))
for dataset, (main, voxel) in tables.items():
    methods = sorted({m for key, rows in reports.items() if key[0] == dataset for m in rows.method.unique()})
    for title, cols in [("Main results", main), ("Voxel errors by spatial scale", voxel)]:
        selected = [label for label in cols if any(fnmatchcase(label, pattern) for pattern in TABLE_COLUMNS)]
        if any(cols[label] is not None for label in selected):
            render(f"{dataset} · {title}", cols, selected, methods)

In [ ]:
import sys
from types import SimpleNamespace
from itertools import islice
sys.path.insert(0, str(root))
from benchmarks import run_metric as runner

pred = PRED_ROOT / DATASET / SPLIT
if not pred.is_dir():
    raise FileNotFoundError(pred)
folders = sorted(p for p in pred.iterdir() if p.is_dir())
selected = [pred / name for name in METHODS] if METHODS is not None else folders
names = sorted({f.parent.name for folder in selected for f in folder.glob("*/events.npz")})
if SEQUENCE is None and not names:
    raise ValueError("Set SEQUENCE or choose a method with saved events.")
name = SEQUENCE or names[0]
data = Path(DATA_ROOT) if DATA_ROOT else runner.v2.old.setting("EVTWEEN_DATA", runner.v2.old.LAB / "evtween/data")
args = SimpleNamespace(dataset=DATASET, split=SPLIT, data_root=DATA_ROOT)
seq = runner.Sequence(runner.source_for(name, data / DATASET / SPLIT, folders, args))
frames, meta = seq.geometry
if not (0 <= START_FRAME < len(frames)-1 and N_FRAMES >= 2 and np.isfinite(RATE_BIN_S) and RATE_BIN_S > 0):
    raise ValueError("Check START_FRAME, N_FRAMES (at least 2) and RATE_BIN_S.")
last = min(START_FRAME + N_FRAMES - 1, len(frames)-1)
lo, hi = frames[[START_FRAME, last]]
print(f"{DATASET}/{SPLIT}/{name} | frames {START_FRAME}–{last} ({last-START_FRAME+1} frames) | [{lo:.6f}, {hi:.6f}) s")

def bin_edges(a, b):
    edges = np.arange(a, b, RATE_BIN_S)
    return np.append(edges[edges < b], b)

windows = {"Full sequence": bin_edges(frames[0], frames[-1]), "Selected frames": bin_edges(lo, hi)}
rates = {key: {} for key in windows}
polarity_rates = {key: {} for key in windows}
polarity_counts = {key: {} for key in windows}
if not isinstance(RASTER_MAX_POINTS, int) or RASTER_MAX_POINTS < 1:
    raise ValueError("RASTER_MAX_POINTS must be a positive integer.")
if CLOUD_MAX_POINTS is not None and (isinstance(CLOUD_MAX_POINTS, bool) or
        not isinstance(CLOUD_MAX_POINTS, int) or CLOUD_MAX_POINTS < 1):
    raise ValueError("CLOUD_MAX_POINTS must be a positive integer or None.")
raster = {}
clouds = {}
maps = {}
shape = meta["height"], meta["width"]

def collect(method, stream):
    t = stream["t"]
    on_times = t[stream["p"] > 0]
    for key, edges in windows.items():
        counts = np.diff(np.searchsorted(t, edges, side="left"))
        on = np.diff(np.searchsorted(on_times, edges, side="left"))
        off = counts - on
        widths = np.diff(edges)
        rates[key][method] = counts / widths
        polarity_rates[key][method] = (on / widths, off / widths)
        polarity_counts[key][method] = (int(on.sum()), int(off.sum()))
    a, b = np.searchsorted(t, [lo, hi])
    count = int(b - a)
    indices = np.arange(a, b) if count <= RASTER_MAX_POINTS else (
        a + np.sort(np.random.default_rng(0).choice(count, RASTER_MAX_POINTS, replace=False)))
    raster[method] = dict(t=t[indices].tolist(), x=stream["x"][indices].tolist(),
                          on=(stream["p"][indices] > 0).tolist(), total=count)
    # Keep a bounded uniform sample; the export cell applies one shared fraction.
    n_cloud = count if CLOUD_MAX_POINTS is None else min(count, CLOUD_MAX_POINTS)
    cloud_indices = (np.arange(a, b) if n_cloud == count else
                     a + np.sort(np.random.default_rng(CLOUD_SEED).choice(
                         count, n_cloud, replace=False)))
    clouds[method] = dict(t=t[cloud_indices], x=stream["x"][cloud_indices],
                          y=stream["y"][cloud_indices], on=stream["p"][cloud_indices] > 0,
                          total=count, total_on=int(np.count_nonzero(stream["p"][a:b] > 0)))
    xy = stream["y"][a:b].astype(np.int64)*shape[1] + stream["x"][a:b]
    maps[method] = np.bincount(xy, weights=np.where(stream["p"][a:b] > 0, 1, -1),
                              minlength=shape[0]*shape[1]).reshape(shape)

collect("Ground truth", seq.gt)
for folder in selected:
    native = folder.name == "v2v"
    path = folder / name / ("voxels.npz" if native else "events.npz")
    if not path.is_file():
        print(f"Missing {folder.name}: {path}")
        continue
    runner.verify_metadata(path.parent / "input.json", meta, args, name)
    if native:
        raster[folder.name] = None
        clouds[folder.name] = None
        with runner.native_planes(path, frames, *shape) as planes:
            maps[folder.name] = sum(islice(planes, START_FRAME, last), np.zeros(shape))
        print("v2v: native signed voxels are shown in the map; individual timestamps are unavailable.")
    else:
        collect(folder.name, runner.read_events(path, meta))

In [ ]:
# Signed count maps on identical coordinates and one shared color scale.
from PIL import Image
from io import BytesIO
import base64

peaks = [np.percentile(np.abs(m[m != 0]), 99) for m in maps.values() if np.any(m)]
limit = max([1.0, *peaks])
panels = []
for method, counts in maps.items():
    scaled = np.clip(counts / limit, -1, 1)
    rgb = np.stack([1 + np.minimum(scaled, 0), 1 - np.abs(scaled), 1 - np.maximum(scaled, 0)], axis=-1)
    buffer = BytesIO()
    Image.fromarray((255*rgb).astype("uint8")).save(buffer, format="PNG")
    src = base64.b64encode(buffer.getvalue()).decode()
    panels.append(f'<figure style="margin:5px"><figcaption>{escape(method)}</figcaption>'
                  f'<img style="width:100%" src="data:image/png;base64,{src}"></figure>')
print(f"Frames {START_FRAME}–{last} | ON minus OFF per pixel | red: positive, blue: negative, white: zero | saturation ±{limit:.1f}")
display(HTML('<div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(235px,1fr))">' + "".join(panels) + '</div>'))

In [ ]:
# Full event counts in this window, before display sampling.
PLOT_WINDOW = "Selected frames"  # Or "Full sequence".
counts = pd.DataFrame.from_dict(polarity_counts[PLOT_WINDOW], orient="index",
                                columns=["ON events", "OFF events"])
counts.index.name = "Method"
display(counts.assign(**{"Total events": counts.sum(axis=1)}))

### 1. ON and OFF event rates
The two panels use all events and share their time and rate scales. Click model names to toggle both panels. Drag to zoom or double-click to reset. Choose Total for the combined event rate.

In [ ]:
payload = {key: {"edges": edges.tolist(), "rates": {m: [on.tolist(), off.tolist()] for m, (on, off) in polarity_rates[key].items()}}
           for key, edges in windows.items()}
title = f"{DATASET}/{SPLIT}/{name} · frames {START_FRAME}–{last} · {RATE_BIN_S*1000:g} ms bins"
document = """<!doctype html><meta charset="utf-8"><title>Event rate comparison</title>
<style>*{box-sizing:border-box}body{font:14px system-ui;color:#192a3a;background:white;margin:16px}
button,select{font:inherit;margin:3px;padding:4px 8px;background:white;border:1px solid #aab4bb;border-radius:4px}
button[aria-pressed=false]{opacity:.42}canvas{width:100%;height:350px;touch-action:none}output{display:block;min-height:35px}</style>
<h3>Event rate over time</h3><p id="info"></p><select id="scope"><option>Selected frames</option><option>Full sequence</option></select>
<select id="mode"><option>ON and OFF</option><option>Total</option></select><button id="reset">Reset zoom</button><div id="legend"></div><canvas id="chart"></canvas><canvas id="second"></canvas><output id="readout"></output>
<script>
const data=__DATA__, title=__TITLE__;
const el=id=>document.getElementById(id), canvases=[el('chart'),el('second')];
el('info').textContent=title;
const names=Object.keys(data['Selected frames'].rates), active=new Set(names);
const color=i=>i===0?'#151515':'hsl('+((i*137)%360)+' 68% 40%)';
let left,right,start=null,L=76,R,B=302,T=18;
const current=()=>data[el('scope').value];
function restore(){const e=current().edges; left=e[0];right=e[e.length-1];el('readout').textContent='';draw()}
function bin(e,t){let a=0,b=e.length-1;while(a+1<b){let m=(a+b)>>1;if(e[m]<=t)a=m;else b=m}return a}
function draw(){const total=el('mode').value==='Total';canvases[1].hidden=total;
canvases.slice(0,total?1:2).forEach((canvas,panel)=>{const ctx=canvas.getContext('2d');let w=canvas.clientWidth,h=350,k=devicePixelRatio||1;canvas.width=w*k;canvas.height=h*k;ctx.scale(k,k);R=w-20;
const {edges:e,rates}=current(),a=bin(e,left),b=bin(e,right);let ymax=1;
for(const n of active)for(let i=a;i<=b;i++)ymax=total?Math.max(ymax,rates[n][0][i]+rates[n][1][i]):Math.max(ymax,rates[n][0][i],rates[n][1][i]);
const X=t=>L+(t-left)/(right-left)*(R-L),Y=v=>B-v/ymax*(B-T);
ctx.font='11px system-ui';ctx.fillStyle='#334';ctx.strokeStyle='#ddd';
for(let j=0;j<=4;j++){let y=Y(ymax*j/4);ctx.beginPath();ctx.moveTo(L,y);ctx.lineTo(R,y);ctx.stroke();ctx.fillText((ymax*j/4).toExponential(1),3,y+4)}
for(let j=0;j<=5;j++)ctx.fillText((left+(right-left)*j/5).toFixed(4),L+(R-L)*j/5-18,B+17);
ctx.fillText((total?'Total':panel?'OFF':'ON')+' events/s',4,12);ctx.fillText('Time since evaluation start (s)',Math.max(L,w/2-73),B+39);
ctx.save();ctx.beginPath();ctx.rect(L,T,R-L,B-T);ctx.clip();
names.forEach((n,j)=>{if(!active.has(n))return;ctx.strokeStyle=color(j);ctx.lineWidth=j===0?2.4:1.4;ctx.beginPath();
for(let i=a;i<=b;i++){let y=Y(total?rates[n][0][i]+rates[n][1][i]:rates[n][panel][i]);if(i===a)ctx.moveTo(X(e[i]),y);else ctx.lineTo(X(e[i]),y);ctx.lineTo(X(e[i+1]),y)}ctx.stroke()});ctx.restore()})}
names.forEach((n,j)=>{const btn=document.createElement('button');btn.textContent=n;btn.style.borderLeft='5px solid '+color(j);btn.setAttribute('aria-pressed','true');
btn.onclick=()=>{if(active.has(n))active.delete(n);else active.add(n);btn.setAttribute('aria-pressed',String(active.has(n)));draw()};el('legend').append(btn)});
function time(ev){let x=ev.clientX-ev.currentTarget.getBoundingClientRect().left;return left+Math.max(0,Math.min(1,(x-L)/(R-L)))*(right-left)}
canvases.forEach(canvas=>{canvas.onpointerdown=ev=>{start=time(ev);canvas.setPointerCapture(ev.pointerId)};
canvas.onpointerup=ev=>{if(start===null)return;const end=time(ev);if(Math.abs(end-start)>(right-left)*.01){left=Math.min(start,end);right=Math.max(start,end);draw()}start=null};
canvas.onpointermove=ev=>{const {edges:e,rates}=current(),i=bin(e,time(ev));el('readout').textContent='['+e[i].toFixed(5)+', '+e[i+1].toFixed(5)+') s   '+[...active].map(n=>n+': '+Math.round(rates[n][0][i]).toLocaleString()+' ON/s, '+Math.round(rates[n][1][i]).toLocaleString()+' OFF/s').join(' | ')};
canvas.ondblclick=restore});el("mode").onchange=draw;el('reset').onclick=restore;el('scope').onchange=restore;window.onresize=draw;restore();
</script>"""
document = document.replace("__TITLE__", json.dumps(title).replace("<", "\\u003c")).replace(
    "__DATA__", json.dumps(payload, separators=(",", ":"), allow_nan=False).replace("<", "\\u003c"))
output = REPORT_ROOT / "visualizations" / DATASET / SPLIT / name / f"rate_frames_{START_FRAME}_{last}.html"
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(document, encoding="utf-8")
# Embed the same self-contained HTML even when Jupyter serves only the notebook directory.
display(HTML('<iframe title="Event rate comparison" style="width:100%;height:1020px;border:0" srcdoc="' +
             escape(document, quote=True) + '"></iframe>'))
print("Standalone HTML:", output)


In [ ]:
# 2. x versus time: one point per displayed event, all methods on the same axes.
raster_page = """<!doctype html><meta charset="utf-8"><title>Event x versus time</title>
<style>body{font:14px system-ui;color:#192a3a;background:white;margin:16px}button{padding:5px}main{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,315px),1fr));gap:12px}canvas{width:100%;height:210px;touch-action:none}h4{margin:12px 0 0}</style>
<h3>Event x versus time</h3><p id="info"></p>
<label><input id="on" type="checkbox" checked>ON (red)</label>
<label><input id="off" type="checkbox" checked>OFF (blue)</label>
<button id="reset">Reset time range</button><p>Drag across any panel to zoom all panels.</p><main></main>
<script>
const {data,lo,hi,width,title}=__RASTER__;
document.getElementById('info').textContent=title;
let left=lo,right=hi;
const drawPanels=[];
function redraw(){drawPanels.forEach(draw=>draw())}
function reset(){left=lo;right=hi;redraw()}
for(const [method,sample] of Object.entries(data)){
  const panel=document.createElement('section'), heading=document.createElement('h4');
  heading.textContent=method;panel.append(heading);document.querySelector('main').append(panel);
  if(!sample){panel.append('Native signed voxels: individual timestamps unavailable.');continue}
  const {t,x,on,total}=sample, canvas=document.createElement('canvas');
  panel.append(canvas);
  function draw(){
    const w=canvas.clientWidth,h=210,k=window.devicePixelRatio||1;
    canvas.width=w*k;canvas.height=h*k;const ctx=canvas.getContext('2d');ctx.scale(k,k);
    const L=44,R=w-14,T=16,B=h-40;
    ctx.font='11px system-ui';ctx.fillStyle='#263b49';ctx.strokeStyle='#bbb';ctx.strokeRect(L,T,R-L,B-T);
    for(let i=0;i<=4;i++){
      const frac=i/4;ctx.fillText(((left+(right-left)*frac).toFixed(3)),L+(R-L)*frac-14,B+14);
      ctx.fillText(Math.round((width-1)*frac),4,B-(B-T)*frac+4)
    }
    ctx.fillText('x (px)',3,11);ctx.fillText('Time since evaluation start (s)',L,B+33);
    const showOn=document.getElementById('on').checked,showOff=document.getElementById('off').checked;
    let visible=0;
    for(let i=0;i<t.length;i++){
      if(t[i]<left || t[i]>=right || (on[i]?!showOn:!showOff))continue;
      ctx.fillStyle=on[i]?'#e23232':'#2150d4';
      ctx.fillRect(L+(t[i]-left)/(right-left)*(R-L-1.6),B-1.6-x[i]/Math.max(1,width-1)*(B-T-1.6),1.6,1.6);
      visible++;
    }
    heading.textContent=method+' | '+visible.toLocaleString()+' shown / '+total.toLocaleString()+' in frame window';
  }
  drawPanels.push(draw);
  function time(e){const f=(e.clientX-canvas.getBoundingClientRect().left-44)/(canvas.clientWidth-58);
    return left+Math.min(1,Math.max(0,f))*(right-left)}
  let start=null;canvas.onpointerdown=e=>{start=time(e);canvas.setPointerCapture(e.pointerId)};
  canvas.onpointerup=e=>{const end=time(e);if(start!==null && Math.abs(end-start)>(right-left)*.01){
    left=Math.min(start,end);right=Math.max(start,end);redraw()}start=null};
  canvas.ondblclick=reset;
}
document.getElementById('reset').onclick=reset;
for(const id of ['on','off'])document.getElementById(id).onchange=redraw;
window.onresize=redraw;redraw();
</script>"""
context = dict(data=raster, lo=float(lo), hi=float(hi), width=int(meta["width"]),
               title=f"{DATASET}/{SPLIT}/{name} · frames {START_FRAME}–{last} · deterministic display sample")
raster_page = raster_page.replace("__RASTER__", json.dumps(context, separators=(",", ":"),
                                  allow_nan=False).replace("<", "\\u003c"))
raster_file = REPORT_ROOT / "visualizations" / DATASET / SPLIT / name / f"x_time_frames_{START_FRAME}_{last}.html"
raster_file.parent.mkdir(parents=True, exist_ok=True)
raster_file.write_text(raster_page, encoding="utf-8")
display(HTML('<iframe title="Event x versus time" style="width:100%;height:830px;border:0" srcdoc="' +
             escape(raster_page, quote=True) + '"></iframe>'))
print("Standalone HTML:", raster_file)


### 3. Interactive 3D event clouds (x, y, time)
Ground truth and all selected event-generating models share the same coordinate ranges and camera.
Drag any panel to rotate all clouds, scroll to zoom, or Shift-drag to pan. Use XY, X-time,
and Y-time views for aligned projections. Filter ON/OFF, hide models, and select a time range.
Time is in **milliseconds relative to the selected window start**; spatial coordinates are pixels
in the same benchmark crop as the plots above. Geometry is displayed in a normalized cube
(pixels and milliseconds are different units); there is no nearest-event time rescaling.

To keep the HTML responsive, all methods use the **same display sampling fraction**, determined
by `CLOUD_MAX_POINTS` and the largest event count. Counts, rate plots, and saved events remain
unchanged. Panel headers distinguish the complete-window counts from the displayed sample.
Narrowing the time range filters that saved sample; increase the cap and rerun the loading cell
to inspect more events. `CLOUD_MAX_POINTS = None` embeds every event and may create a large HTML.
Native voxel-only outputs have no event cloud. The saved HTML is self-contained and works offline.


In [ ]:
# Export the same interactive cloud comparison shown below; no extra plotting dependency.
cloud_counts = [sample["total"] for sample in clouds.values() if sample is not None]
largest_cloud = max(cloud_counts, default=0)
cloud_fraction = (1.0 if CLOUD_MAX_POINTS is None or largest_cloud == 0 else
                  min(1.0, CLOUD_MAX_POINTS / largest_cloud))
cloud_data = {}
for method, sample in clouds.items():
    if sample is None:
        cloud_data[method] = None
        continue
    # A second uniform subset gives each method the same expected display density.
    n_show = (sample["total"] if cloud_fraction == 1 else
              min(len(sample["t"]), int(np.floor(sample["total"] * cloud_fraction))))
    indices = (np.arange(len(sample["t"])) if n_show == len(sample["t"]) else
               np.sort(np.random.default_rng(CLOUD_SEED + 1).choice(
                   len(sample["t"]), n_show, replace=False)))
    cloud_data[method] = dict(
        t_ms=((sample["t"][indices] - lo) * 1000).tolist(),
        x=sample["x"][indices].tolist(), y=sample["y"][indices].tolist(),
        on=sample["on"][indices].tolist(), total=sample["total"], total_on=sample["total_on"])

cloud_context = dict(
    data=cloud_data, duration_ms=float((hi-lo)*1000), start_s=float(lo),
    width=int(meta["width"]), height=int(meta["height"]), fraction=cloud_fraction,
    seed=CLOUD_SEED, max_points=CLOUD_MAX_POINTS,
    title=f"{DATASET}/{SPLIT}/{name} · frames {START_FRAME}–{last}")
cloud_page = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Interactive 3D event clouds</title>
<style>
*{box-sizing:border-box}body{font:14px system-ui,sans-serif;color:#192a3a;background:#fff;margin:18px}
h2{font-size:21px;margin:0 0 8px}p{line-height:1.5;margin:8px 0}
button,input{font:inherit}button{padding:6px 10px;background:white;border:1px solid #aab4bb;border-radius:5px;cursor:pointer}
button:hover{background:#edf4f9}button:focus-visible,input:focus-visible{outline:2px solid #167bc1;outline-offset:2px}
.controls{display:flex;gap:9px;align-items:center;flex-wrap:wrap;margin:10px 0}
.controls label{display:inline-flex;gap:5px;align-items:center}.controls input[type=range]{width:160px}
input[type=number]{width:108px;padding:4px}input[type=checkbox]{accent-color:#24669a}
.note{color:#506170;font-size:12px}.legend-on{color:#c72b34}.legend-off{color:#245bcc}
#methods{border-top:1px solid #dae2e8;padding-top:12px}
main{display:grid;grid-template-columns:repeat(auto-fit,minmax(min(100%,330px),1fr));gap:14px;margin-top:16px}
section{min-width:0;border:1px solid #dce4eb;border-radius:8px;overflow:hidden;background:#fbfcfe}
section h3{font-size:14px;margin:12px 12px 6px;overflow-wrap:anywhere}
.counts,.readout{font-size:12px;margin:5px 12px;min-height:32px;line-height:1.4;color:#455968}
.readout{min-height:34px}canvas{display:block;width:100%;height:310px;touch-action:none;cursor:grab}
canvas:active{cursor:grabbing}section[hidden]{display:none}.unavailable{padding:18px 12px;line-height:1.5}
</style>
</head>
<body>
<h2>3D event clouds: ground truth and generated events</h2>
<p id="info"></p>
<p class="note">Drag to rotate, scroll to zoom, Shift-drag to pan. Views are linked across panels.
Hover for event coordinates. Spatial axes use pixels; time uses milliseconds relative to the window start.
The display cube normalizes these different units; it is not a distance metric.</p>
<div class="controls" aria-label="Polarity and view controls">
<label class="legend-on"><input id="on" type="checkbox" checked>ON (red)</label>
<label class="legend-off"><input id="off" type="checkbox" checked>OFF (blue)</label>
<button id="view3d">3D / reset view</button><button id="xy">XY</button>
<button id="xt">X-time</button><button id="yt">Y-time</button>
<label>Point size <input id="size" type="range" min="1" max="4" step="0.5" value="1.5"></label>
</div>
<div class="controls" aria-label="Time range">
<label>From (ms) <input id="start" type="number" step="any" value="0"></label>
<input id="startSlider" aria-label="Time range start" type="range" min="0" max="1000" value="0">
<label>To (ms) <input id="end" type="number" step="any"></label>
<input id="endSlider" aria-label="Time range end" type="range" min="0" max="1000" value="1000">
<button id="timeReset">Full time range</button>
</div>
<p id="sampling" class="note"></p>
<div id="methods" class="controls" aria-label="Visible methods"></div>
<main id="panels"></main>
<script>
'use strict';
const payload=__CLOUD_DATA__;
const {data,duration_ms:duration,width,height,fraction}=payload;
const el=id=>document.getElementById(id), fmt=n=>n.toLocaleString();
el('info').textContent=payload.title+' | window start '+payload.start_s.toFixed(6)+' s | '+duration.toFixed(3)+' ms';
el('sampling').textContent='Shared display fraction: '+(fraction*100).toFixed(3)+'%. '
  +'Full-window counts are exact. Visible counts refer only to sampled points after time/polarity filters. '
  +'Time filtering does not load more points. Seed: '+payload.seed+'.';
let yaw=-0.65,pitch=0.38,zoom=1,panX=0,panY=0,left=0,right=duration;
let pending=false;
const panels=[];
function schedule(){if(!pending){pending=true;requestAnimationFrame(()=>{pending=false;panels.forEach(p=>p.draw())})}}
function setView(y,p){yaw=y;pitch=p;zoom=1;panX=panY=0;schedule()}
el('view3d').onclick=()=>setView(-0.65,0.38);
el('xy').onclick=()=>setView(0,0);
el('xt').onclick=()=>setView(0,Math.PI/2);
el('yt').onclick=()=>setView(Math.PI/2,0);
function timeControls(){
  el('start').value=String(left);el('end').value=String(right);
  el('startSlider').value=left/duration*1000;el('endSlider').value=right/duration*1000;
  el('start').min=0;el('start').max=duration;el('end').min=0;el('end').max=duration;
}
function changeTime(start,value){
  if(!Number.isFinite(value)){timeControls();return}
  value=Math.max(0,Math.min(duration,value));
  if(start){left=Math.min(value,right)}else{right=Math.max(value,left)}
  timeControls();schedule();
}
el('start').onchange=()=>changeTime(true,el('start').valueAsNumber);
el('end').onchange=()=>changeTime(false,el('end').valueAsNumber);
el('startSlider').oninput=()=>changeTime(true,Number(el('startSlider').value)*duration/1000);
el('endSlider').oninput=()=>changeTime(false,Number(el('endSlider').value)*duration/1000);
el('timeReset').onclick=()=>{left=0;right=duration;timeControls();schedule()};
for(const id of ['on','off','size'])el(id).oninput=schedule;
timeControls();
const corners=[[-1,-1,-1],[1,-1,-1],[1,1,-1],[-1,1,-1],[-1,-1,1],[1,-1,1],[1,1,1],[-1,1,1]];
const edges=[[0,1],[1,2],[2,3],[3,0],[4,5],[5,6],[6,7],[7,4],[0,4],[1,5],[2,6],[3,7]];
for(const [method,sample] of Object.entries(data)){
  const section=document.createElement('section'),heading=document.createElement('h3');
  heading.textContent=method;section.append(heading);el('panels').append(section);
  const label=document.createElement('label'),toggle=document.createElement('input');
  toggle.type='checkbox';toggle.checked=true;label.append(toggle,document.createTextNode(method));el('methods').append(label);
  toggle.onchange=()=>{section.hidden=!toggle.checked;schedule()};
  if(sample===null){
    const note=document.createElement('p');note.className='unavailable';
    note.textContent='Native signed voxels: individual event coordinates and timestamps are unavailable.';
    section.append(note);continue;
  }
  const {t_ms:t,x,y,on,total,total_on:totalOn}=sample;
  const counts=document.createElement('div'),canvas=document.createElement('canvas'),readout=document.createElement('div');
  counts.className='counts';readout.className='readout';canvas.setAttribute('aria-label',method+' 3D event cloud');
  section.append(counts,canvas,readout);
  // Keep projected positions for hover; no network, external assets or notebook kernel is used.
  const sx=new Float32Array(t.length),sy=new Float32Array(t.length),visible=new Uint8Array(t.length);
  let drag=null,drawWidth=1,drawHeight=310;
  function draw(){
    if(section.hidden)return;
    const w=canvas.clientWidth,h=310,dpr=Math.min(2,window.devicePixelRatio||1);
    if(w<1)return;
    drawWidth=w;drawHeight=h;canvas.width=Math.round(w*dpr);canvas.height=Math.round(h*dpr);
    const ctx=canvas.getContext('2d');ctx.scale(dpr,dpr);
    const cy=Math.cos(yaw),siy=Math.sin(yaw),cp=Math.cos(pitch),sp=Math.sin(pitch);
    const scale=Math.min(w,h)*0.245*zoom,cx=w/2+panX*w,centerY=h/2+panY*h;
    function project(a,b,c){
      const rx=cy*a+siy*c,rz=-siy*a+cy*c,ry=cp*b-sp*rz;
      return [cx+scale*rx,centerY-scale*ry];
    }
    const box=corners.map(v=>project(...v));
    ctx.lineWidth=1;ctx.strokeStyle='#d8e0e7';ctx.beginPath();
    for(const [a,b] of edges){ctx.moveTo(...box[a]);ctx.lineTo(...box[b])}ctx.stroke();
    const showOn=el('on').checked,showOff=el('off').checked,size=Number(el('size').value);
    let shown=0,shownOn=0;visible.fill(0);
    ctx.globalAlpha=0.7;
    for(let i=0;i<t.length;i++){
      if(t[i]<left||t[i]>=right||(on[i]?!showOn:!showOff))continue;
      const [px,py]=project(2*x[i]/Math.max(1,width-1)-1,1-2*y[i]/Math.max(1,height-1),2*t[i]/duration-1);
      sx[i]=px;sy[i]=py;visible[i]=1;shown++;if(on[i])shownOn++;
      ctx.fillStyle=on[i]?'#d52d37':'#245bcc';ctx.fillRect(px-size/2,py-size/2,size,size);
    }
    ctx.globalAlpha=1;ctx.font='11px system-ui';
    const axes=[{a:[-1,1,-1],b:[1,1,-1],text:'x (px)',max:width-1,color:'#263d52'},
      {a:[-1,1,-1],b:[-1,-1,-1],text:'y (px)',max:height-1,color:'#263d52'},
      {a:[-1,1,-1],b:[-1,1,1],text:'time (ms)',max:duration,color:'#30674d'}];
    for(const axis of axes){
      const a=project(...axis.a),b=project(...axis.b),dx=b[0]-a[0],dy=b[1]-a[1],length=Math.hypot(dx,dy);
      if(length<25)continue;
      ctx.strokeStyle=axis.color;ctx.fillStyle=axis.color;ctx.lineWidth=1.5;
      ctx.beginPath();ctx.moveTo(...a);ctx.lineTo(...b);ctx.stroke();
      const nx=-dy/length,ny=dx/length;
      for(const f of [0,0.5,1]){
        const px=a[0]+dx*f,py=a[1]+dy*f;
        ctx.beginPath();ctx.moveTo(px,py);ctx.lineTo(px+nx*4,py+ny*4);ctx.stroke();
        const value=axis.max*f,txt=axis.text==='time (ms)'?value.toFixed(2):String(Math.round(value));
        ctx.fillText(txt,px+nx*10,py+ny*10);
      }
      ctx.fillText(axis.text,(a[0]+b[0])/2+nx*25,(a[1]+b[1])/2+ny*25);
    }
    counts.textContent='Full window: '+fmt(total)+' events ('+fmt(totalOn)+' ON / '+fmt(total-totalOn)+' OFF). '
      +'Visible sample: '+fmt(shown)+' / '+fmt(t.length)+' embedded ('+fmt(shownOn)+' ON / '+fmt(shown-shownOn)+' OFF).';
    if(total===0)readout.textContent='No events in the selected frame window.';
    else if(t.length===0)readout.textContent='No events retained at this display fraction. Increase CLOUD_MAX_POINTS.';
    else if(shown===0)readout.textContent='No sampled events in this time/polarity selection.';
    else readout.textContent='Hover over a point to inspect x, y, time and polarity.';
  }
  panels.push({draw});
  canvas.onpointerdown=e=>{
    if(e.button!==0)return;
    drag={x:e.clientX,y:e.clientY,yaw,pitch,panX,panY,pan:e.shiftKey,id:e.pointerId};
    canvas.setPointerCapture(e.pointerId);
  };
  canvas.onpointermove=e=>{
    if(drag){
      const dx=e.clientX-drag.x,dy=e.clientY-drag.y;
      if(drag.pan){panX=drag.panX+dx/drawWidth;panY=drag.panY+dy/drawHeight}
      else{yaw=drag.yaw+dx*0.008;pitch=Math.max(-Math.PI/2,Math.min(Math.PI/2,drag.pitch+dy*0.008))}
      schedule();return;
    }
    const bounds=canvas.getBoundingClientRect(),mx=e.clientX-bounds.left,my=e.clientY-bounds.top;
    let nearest=-1,distance=64;
    for(let i=0;i<t.length;i++){
      if(!visible[i])continue;
      const d=(sx[i]-mx)**2+(sy[i]-my)**2;if(d<distance){distance=d;nearest=i}
    }
    readout.textContent=nearest<0?'Hover over a point to inspect x, y, time and polarity.':
      'x='+x[nearest]+' px | y='+y[nearest]+' px | t='+t[nearest].toFixed(5)+' ms | '+(on[nearest]?'ON (+)':'OFF (-)');
  };
  const release=e=>{if(canvas.hasPointerCapture(e.pointerId))canvas.releasePointerCapture(e.pointerId);drag=null};
  canvas.onpointerup=release;canvas.onpointercancel=release;canvas.onlostpointercapture=()=>{drag=null};
  canvas.addEventListener('wheel',e=>{e.preventDefault();zoom=Math.max(0.25,Math.min(8,zoom*Math.exp(-e.deltaY*0.001)));schedule()},{passive:false});
  canvas.ondblclick=()=>setView(-0.65,0.38);
}
window.addEventListener('resize',schedule);schedule();
</script>
</body>
</html>
"""
cloud_page = cloud_page.replace("__CLOUD_DATA__", json.dumps(
    cloud_context, separators=(",", ":"), allow_nan=False).replace("<", "\\u003c"))
cloud_file = (REPORT_ROOT / "visualizations" / DATASET / SPLIT / name /
              f"event_cloud_frames_{START_FRAME}_{last}.html")
cloud_file.parent.mkdir(parents=True, exist_ok=True)
cloud_file.write_text(cloud_page, encoding="utf-8")
display(HTML('<iframe title="Interactive 3D event clouds" '
             'style="width:100%;height:1050px;border:0" srcdoc="' +
             escape(cloud_page, quote=True) + '"></iframe>'))
print(f"Cloud display fraction: {cloud_fraction:.3%} (full event counts are unchanged)")
print("Standalone HTML:", cloud_file)
